# Moral foundation coding with an OpenAI model — main analysis

The baseline cell: **main prompt × 0-shot**, model `gpt-6-luna` (non-reasoning, `reasoning_effort="none"`).

Before running:
* Select the Python kernel (top right) you want to use.
* Provide your OpenAI API key (prompted below, or set `OPENAI_API_KEY` in a `.env` file — keep `.env` out of git).

## Setup

In [1]:
# Run once, then restart the kernel
%pip install -q -U openai pandas tqdm scikit-learn ipywidgets python-dotenv

^C
ERROR: Operation cancelled by user
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Reload run_grid_openai.py automatically whenever you edit it
%load_ext autoreload
%autoreload 2

In [3]:
import os, json, getpass
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, cohen_kappa_score

try:
    from dotenv import load_dotenv
    load_dotenv()                      # reads OPENAI_API_KEY from .env if present
except ImportError:
    pass
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

import run_grid_openai as rg
print("model:", rg.MODEL, "| reasoning_effort:", rg.REASONING_EFFORT, "| output folder:", rg.OUT_DIR)

model: gpt-6-luna | reasoning_effort: none | output folder: runs/gpt-6-luna


## Load validation set

In [4]:
validation = rg.load_validation()
print(validation.shape)
validation.head()

(250, 20)


,para_id,text,moral_Yunfan,care_Yunfan,fairness_Yunfan,loyalty_Yunfan,authority_Yunfan,sanctity_Yunfan,moral_Wiktoria,care_Wiktoria,fairness_Wiktoria,loyalty_Wiktoria,authority_Wiktoria,sanctity_Wiktoria,moral_Carlos,care_Carlos,fairness_Carlos,loyalty_Carlos,authority_Carlos,sanctity_Carlos
0,CREC-2025-03-31-pt2-PgS2034-2_295_46,He made millions and millions of dollars--from...,1,0,0,0,1,0,1,0,0,0,1,0,1,0,0,0,1,0
1,CREC-2026-02-12-pt1-PgS593_34_19,You have an obligation under Federal law to tu...,1,0,0,0,1,0,1,0,0,0,1,0,1,0,0,0,1,0
2,CREC-2025-04-28-pt1-PgH1673_5_21,I refuse to go backward. I am determined to co...,1,0,0,1,0,0,1,1,0,0,0,0,1,1,0,1,0,0
3,CREC-2026-02-24-pt1-PgH2262-3_2_3,"She made sure every single meeting, policy dec...",1,1,0,0,0,0,1,1,0,0,1,0,1,1,0,0,0,0
4,CREC-2025-12-02-pt1-PgH4978-2_16_16,As a longtime mediocre swimmer and a former sw...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Prompt
The main prompt is identical to the one in `llm_moral_coding.ipynb`.

In [5]:
print(rg.make_prompt(validation["text"][0]))



1. TASK
You're an expert political scientist analyzing U.S. congressional floor speeches. You need to decide whether a paragraph from the speeches uses moral rhetoric, and if it does, which moral foundations it draws on.
Code six labels, each as 1 (present) or 0 (absent): MORAL RHETORIC, CARE, FAIRNESS, LOYALTY, AUTHORITY, SANCTITY.

2. CODING APPROACH
* The unit of analysis is the paragraph. Read the full paragraph before coding.
* Code each label for presence (1) or absence (0).
* Code a label as 1 if the paragraph expresses its meaning at least once.
* A paragraph can be coded for several moral foundations at the same time.

Procedure:
1. Read the full paragraph.
2. Code MORAL RHETORIC as 1 or 0.
3. If MORAL RHETORIC is 1, code each of the five moral foundations as 1 or 0. At least one foundation must be 1.
4. If MORAL RHETORIC is 0, all five moral foundations must be 0.

3. LABELS


Label: MORAL RHETORIC
Definition: The speaker, explicitly or implicitly,
* judges an action, actor

Test on one paragraph:

In [6]:
rg.check_params(rg.make_prompt(validation["text"][0]))   # drops temperature/seed if the model rejects them
rg.ask_llm(rg.make_prompt(validation["text"][0]), validation["text"][0])

{'moral_rhetoric': 0,
 'care': 0,
 'fairness': 0,
 'loyalty': 0,
 'authority': 0,
 'sanctity': 0,
 'evidence_moral_rhetoric': '',
 'evidence_care': '',
 'evidence_fairness': '',
 'evidence_loyalty': '',
 'evidence_authority': '',
 'evidence_sanctity': '',
 'problem': None,
 'warnings': [],
 'raw': '{"evidence":{"moral_rhetoric":"","care":"","fairness":"","loyalty":"","authority":"","sanctity":""},"moral_rhetoric":0,"care":0,"fairness":0,"loyalty":0,"authority":0,"sanctity":0}',
 'model_version': 'gpt-6-luna'}

## Run the primary cell (main × 0-shot)

Every paragraph is coded once; no resume logic. Results are saved to
`runs/gpt-6-luna/main_0shot_validation.csv`, together with the exact prompt (`_prompt.txt`)
and the settings used (`_settings.json`).

In [8]:
pred = rg.run_cell(validation, "main", 0)
pred.head()

main, 0-shot:   0%|          | 0/250 [00:00<?, ?it/s]

,para_id,text,variant,shots,moral_rhetoric,care,fairness,loyalty,authority,sanctity,evidence_moral_rhetoric,evidence_care,evidence_fairness,evidence_loyalty,evidence_authority,evidence_sanctity,problem,warnings,raw,model_version
0,CREC-2025-03-31-pt2-PgS2034-2_295_46,He made millions and millions of dollars--from...,main,0,0,0,0,0,0,0,,,,,,,None,[],"{""evidence"":{""moral_rhetoric"":"""",""care"":"""",""fa...",gpt-6-luna
1,CREC-2026-02-12-pt1-PgS593_34_19,You have an obligation under Federal law to tu...,main,0,1,0,0,0,1,0,You have an obligation under Federal law to tu...,,,,You have an obligation under Federal law to tu...,,None,[],"{""evidence"":{""moral_rhetoric"":""You have an obl...",gpt-6-luna
2,CREC-2025-04-28-pt1-PgH1673_5_21,I refuse to go backward. I am determined to co...,main,0,1,1,0,1,0,0,“fight for our healthcare”,“fight for our healthcare”,,“arm in arm with my CBC colleagues and allies”,,,None,[],"{""evidence"":{""moral_rhetoric"":""“fight for our ...",gpt-6-luna
3,CREC-2026-02-24-pt1-PgH2262-3_2_3,"She made sure every single meeting, policy dec...",main,0,1,1,0,1,0,0,"""centered on the needs of the beautiful famili...","""the needs of the ... families""",,"""families that we represent""",,,None,[care: evidence not found in paragraph],"{""evidence"":{""moral_rhetoric"":""\""centered on t...",gpt-6-luna
4,CREC-2025-12-02-pt1-PgH4978-2_16_16,As a longtime mediocre swimmer and a former sw...,main,0,0,0,0,0,0,0,,,,,,,None,[],"{""evidence"":{""moral_rhetoric"":"""",""care"":"""",""fa...",gpt-6-luna


In [9]:
settings = json.load(open(f"{rg.OUT_DIR}/main_0shot_validation_settings.json"))
print("dropped parameters:", settings["dropped_params"])     # [] = temperature and seed were used
print("model versions returned:", settings["model_versions"])
print("params:", settings["params"])

dropped parameters: []
model versions returned: ['gpt-6-luna']
params: {'model': 'gpt-6-luna', 'temperature': 0, 'seed': 502, 'reasoning_effort': 'none'}


## Output checks

In [10]:
print("problems:")
print(pred["problem"].fillna("none").value_counts())
warnings = pred["warnings"].apply(lambda w: w if isinstance(w, list) else [])
print("\nwarnings:")
print(pd.Series([x.split(": ", 1)[1] for ws in warnings for x in ws], dtype=str).value_counts())

problems:
problem
none                        246
moral_without_foundation      3
foundation_without_moral      1
Name: count, dtype: int64

warnings:
evidence not found in paragraph    64
evidence given but coded 0         10
Name: count, dtype: int64


## Evaluation against human labels

In [11]:
EVAL_LABELS = ["moral"] + rg.FOUNDATIONS
CODERS = ["Yunfan", "Wiktoria", "Carlos"]
LLM_COL = {"moral": "moral_rhetoric", **{f: f for f in rg.FOUNDATIONS}}

m = validation.merge(pred.drop(columns=["text"], errors="ignore"), on="para_id")
m = m[m["moral_rhetoric"].notna()]          # drop parse/API errors
for l in EVAL_LABELS:                        # majority vote of the three coders
    m[f"{l}_majority"] = (m[[f"{l}_{c}" for c in CODERS]].sum(axis=1) >= 2).astype(int)

def evaluate(df, human_col_fn, llm_col_map=LLM_COL):
    rows = []
    for l in EVAL_LABELS:
        sub = df[[human_col_fn(l), llm_col_map[l]]].dropna()
        y, yhat = sub.iloc[:, 0].astype(int), sub.iloc[:, 1].astype(int)
        p, r, f, _ = precision_recall_fscore_support(y, yhat, average="binary", zero_division=0)
        rows.append({"label": l, "n": len(sub), "human_n": y.sum(), "llm_n": yhat.sum(),
                     "accuracy": (y == yhat).mean(), "precision": p, "recall": r, "f1": f,
                     "kappa": cohen_kappa_score(y, yhat)})
    return pd.DataFrame(rows).set_index("label").round(3)

all_results = {c: evaluate(m, lambda l, c=c: f"{l}_{c}") for c in CODERS}
all_results["majority"] = evaluate(m, lambda l: f"{l}_majority")
results = pd.concat(all_results, names=["coder", "label"])

print("n =", len(m))
for c, res in all_results.items():
    print(f"{c:>9} macro-F1 (5 foundations):", res.loc[rg.FOUNDATIONS, "f1"].mean().round(3))
results

n = 250
   Yunfan macro-F1 (5 foundations): 0.546
 Wiktoria macro-F1 (5 foundations): 0.479
   Carlos macro-F1 (5 foundations): 0.625
 majority macro-F1 (5 foundations): 0.588


n  human_n  llm_n  accuracy  precision  recall     f1  \
coder    label                                                                
Yunfan   moral      250      122    122     0.792      0.787   0.787  0.787   
         care       250       57     72     0.844      0.625   0.789  0.698   
         fairness   250       47     42     0.852      0.619   0.553  0.584   
         loyalty    250       19     44     0.860      0.318   0.737  0.444   
         authority  250       38     43     0.868      0.558   0.632  0.593   
         sanctity   250       15     29     0.896      0.310   0.600  0.409   
Wiktoria moral      250      129    122     0.796      0.820   0.775  0.797   
         care       250       73     72     0.844      0.736   0.726  0.731   
         fairness   250       43     42     0.868      0.619   0.605  0.612   
         loyalty    250       20     44     0.840      0.273   0.600  0.375   
         authority  250       62     43     0.748      0.488   0.339  0.400   
         sanctity   250       14     29     0.876      0.207   0.429  0.279   
Carlos   moral      250      135    122     0.836      0.885   0.800  0.840   
         care       250       78     72     0.912      0.889   0.821  0.853   
         fairness   250       37     42     0.908      0.667   0.757  0.709   
         loyalty    250       19     44     0.884      0.386   0.895  0.540   
         authority  250       42     43     0.876      0.628   0.643  0.635   
         sanctity   250       12     29     0.900      0.276   0.667  0.390   
majority moral      250      126    122     0.816      0.828   0.802  0.815   
         care       250       67     72     0.868      0.736   0.791  0.763   
         fairness   250       41     42     0.900      0.690   0.707  0.699   
         loyalty    250       17     44     0.868      0.318   0.824  0.459   
         authority  250       39     43     0.872      0.581   0.641  0.610   
         sanctity   250       10     29     0.908      0.276   0.800  0.410   

                    kappa  
coder    label             
Yunfan   moral      0.584  
         care       0.594  
         fairness   0.495  
         loyalty    0.378  
         authority  0.514  
         sanctity   0.358  
Wiktoria moral      0.592  
         care       0.621  
         fairness   0.532  
         loyalty    0.298  
         authority  0.247  
         sanctity   0.220  
Carlos   moral      0.673  
         care       0.791  
         fairness   0.654  
         loyalty    0.485  
         authority  0.561  
         sanctity   0.346  
majority moral      0.632  
         care       0.671  
         fairness   0.639  
         loyalty    0.400  
         authority  0.533  
         sanctity   0.373

In [45]:
results.to_csv(f"{rg.OUT_DIR}/main_0shot_evaluation.csv")

### LLM–human agreement vs. human–human agreement (Cohen's κ)

Is the LLM within the range of agreement among the human coders?

In [12]:
from itertools import combinations

rows = []
for l in EVAL_LABELS:
    hh = [cohen_kappa_score(m[f"{l}_{a}"].astype(int), m[f"{l}_{b}"].astype(int))
          for a, b in combinations(CODERS, 2)]
    lh = [cohen_kappa_score(m[f"{l}_{c}"].astype(int), m[LLM_COL[l]].astype(int)) for c in CODERS]
    rows.append({"label": l,
                 "human-human mean": sum(hh) / 3, "human-human min": min(hh), "human-human max": max(hh),
                 "LLM-human mean": sum(lh) / 3, "LLM-human min": min(lh), "LLM-human max": max(lh)})
pd.DataFrame(rows).set_index("label").round(3)

,human-human mean,human-human min,human-human max,LLM-human mean,LLM-human min,LLM-human max
label,,,,,,
moral,0.595,0.535,0.657,0.616,0.584,0.673
care,0.676,0.649,0.708,0.669,0.594,0.791
fairness,0.505,0.435,0.594,0.560,0.495,0.654
loyalty,0.459,0.305,0.544,0.387,0.298,0.485
authority,0.416,0.285,0.613,0.441,0.247,0.561
sanctity,0.356,0.305,0.413,0.308,0.220,0.358


## Disagreements for error analysis

In [47]:
EVID_COL = {l: f"evidence_{LLM_COL[l]}" for l in EVAL_LABELS}

def find_disagreements(df, coder="majority", outfile=None):
    """coder: 'Yunfan' / 'Wiktoria' / 'Carlos' / 'majority'"""
    def diff(row):
        out = []
        for l in EVAL_LABELS:
            h, p = row[f"{l}_{coder}"], row[LLM_COL[l]]
            if pd.isna(h) or pd.isna(p):
                continue
            if int(h) != int(p):
                out.append(l)
        return out

    df = df.copy()
    df["disagree_on"] = df.apply(diff, axis=1)
    errors = df[df["disagree_on"].str.len() > 0]
    cols = (["para_id", "text", "disagree_on"]
            + [c for l in EVAL_LABELS for c in (f"{l}_{coder}", LLM_COL[l])]
            + [EVID_COL[l] for l in EVAL_LABELS])
    if outfile:
        errors[cols].to_csv(outfile, index=False)
    print(f"{coder}: {len(errors)} / {len(df)} paragraphs with at least one disagreement")
    return errors[cols]

errors = find_disagreements(m, coder="majority",
                            outfile=f"{rg.OUT_DIR}/main_0shot_disagreements_majority.csv")
errors

majority: 19 / 50 paragraphs with at least one disagreement


,para_id,text,disagree_on,moral_majority,moral_rhetoric,care_majority,care,fairness_majority,fairness,loyalty_majority,...,authority_majority,authority,sanctity_majority,sanctity,evidence_moral_rhetoric,evidence_care,evidence_fairness,evidence_loyalty,evidence_authority,evidence_sanctity
1,CREC-2025-11-18-pt1-PgH4718-6_32_0,It is time. They are going to have a reckoning...,"[fairness, loyalty, authority]",1,1,0,0,1,0,0,...,1,0,1,1,“rotten system”,,,“everyone in between is finally waking up agai...,,“rotten system”
5,CREC-2025-11-17-pt1-PgH4672-2_1_0,"Speaking of taxes, the tax on the economy and ...","[moral, loyalty]",1,0,0,0,0,0,1,...,0,0,0,0,,,,,,
6,CREC-2025-12-03-pt1-PgH5001-2_1_0,With only 10 voting days left in this year's c...,"[moral, care, fairness]",1,0,1,0,1,0,0,...,0,0,0,0,,"""avert a full-blown healthcare crisis""",,,,
10,CREC-2025-02-12-pt1-PgS955_0_11,The administration tried to freeze funding for...,"[moral, care]",1,0,1,0,0,0,0,...,0,0,0,0,,,,,,
11,CREC-2025-04-28-pt1-PgH1651_14_4,"However, we must not allow President Trump to ...",[authority],1,1,1,1,0,0,1,...,1,0,0,0,“careless policies”,“put this great progress in jeopardy”; “crash ...,,“stand with the domestic manufacturers they cl...,,
12,CREC-2025-05-14-pt1-PgS2920-5_9_6,I oppose all violence against all police offic...,[authority],1,1,1,1,1,1,0,...,0,1,0,0,I oppose all violence against all police offic...,violence against all police officers,why there seems to be an exemption for the vio...,,the officers who protect us,
13,CREC-2025-06-24-pt1-PgH2894_2_3,"Dave's deep knowledge, steady guidance, and pa...","[moral, care]",0,1,0,1,0,0,0,...,0,0,0,0,passion for serving others,serving others,,,,
14,CREC-2025-06-24-pt1-PgS3501_1_32,"Likewise, we can't expect Putin to end his agg...","[moral, loyalty, authority]",0,1,0,0,0,0,0,...,0,1,0,0,America's abandonment of Ukraine,,,America's abandonment of Ukraine,our own strength atrophy,
17,CREC-2026-01-13-pt1-PgH711_4_30,"Yes, once again, it awaits Senate action. The ...",[sanctity],1,1,0,0,0,0,0,...,1,1,1,0,“We must also defend the rule of law itself.”,,,,“We must also defend the rule of law itself.”,
18,CREC-2026-02-24-pt1-PgS638_0_4,"Now, I have served in the Capitol for over two...","[moral, loyalty]",1,0,0,0,0,0,1,...,0,0,0,0,,,,,,


# Full grid: 3 prompt variants × 3 shot conditions

Needs, in the same folder:
* `fewshot_examples.csv` — the few-shot examples in order (first 2 rows = 2-shot, first 5 = 5-shot).
* `DEFINITIONS_PARAPHRASE` in `run_grid_openai.py` — reviewed paraphrased definitions.

Example paragraphs that are also in the validation set are left out of the 2- and 5-shot cells,
so all cells are compared on the paragraphs they have in common.

In [13]:
examples = rg.load_examples()
examples[["para_id"] + rg.LABELS]
print(rg.make_examples(examples.head(2)))          # check the 2-shot block

Coded examples:

Example 1:
Paragraph: "For me, my pro-life stance is deeply rooted in my Christian faith and belief that all human life is precious, and it is a gift from the almighty God. Defending the unborn has long been one of my highest priorities, and the first term I got elected to Congress, one of my first votes was cast in favor of life."
Answer: {"evidence": {"moral_rhetoric": "all human life is precious", "care": "Defending the unborn has long been one of my highest priorities", "fairness": "", "loyalty": "", "authority": "", "sanctity": "all human life is precious, and it is a gift from the almighty God"}, "moral_rhetoric": 1, "care": 1, "fairness": 0, "loyalty": 0, "authority": 0, "sanctity": 1}

Example 2:
Paragraph: "Every American should be outraged at how the Trump administration is treating anyone who dares to stand in their way. Whether you are a U.S. Senator, a U.S. Citizen, or an immigrant, no one is protected by the Trump regime. It is my job to ensure that my co

In [14]:
# The primary cell (main x 0-shot) was already run above as `pred`, so run the other 8 cells.
# Each is saved as runs/gpt-6-luna/{variant}_{shots}shot_validation.csv
other_cells = [(v, k) for v in rg.VARIANTS for k in rg.SHOTS if (v, k) != rg.PRIMARY]
all_pred = pd.concat([pred, rg.run_grid(validation, cells=other_cells, examples=examples)],
                     ignore_index=True)
all_pred.groupby(["variant", "shots"]).size()

main, 2-shot:   0%|          | 0/250 [00:00<?, ?it/s]

main, 5-shot:   0%|          | 0/250 [00:00<?, ?it/s]

paraphrase, 0-shot:   0%|          | 0/250 [00:00<?, ?it/s]

paraphrase, 2-shot:   0%|          | 0/250 [00:00<?, ?it/s]

paraphrase, 5-shot:   0%|          | 0/250 [00:00<?, ?it/s]

reverse, 0-shot:   0%|          | 0/250 [00:00<?, ?it/s]

reverse, 2-shot:   0%|          | 0/250 [00:00<?, ?it/s]

reverse, 5-shot:   0%|          | 0/250 [00:00<?, ?it/s]

variant     shots
main        0        250
            2        250
            5        250
paraphrase  0        250
            2        250
            5        250
reverse     0        250
            2        250
            5        250
dtype: int64

In [15]:
# Evaluate each model condition against human benchmark and save the results to runs/gpt-6-luna/{variant}_{shots}shot_evaluation.csv
for (v, k), g in all_pred.groupby(["variant", "shots"]):
    mm = validation.merge(g.drop(columns=["text"], errors="ignore"), on="para_id")
    mm = mm[mm["moral_rhetoric"].notna()]
    for l in EVAL_LABELS:
        mm[f"{l}_majority"] = (mm[[f"{l}_{c}" for c in CODERS]].sum(axis=1) >= 2).astype(int)
    res = {c: evaluate(mm, lambda l, c=c: f"{l}_{c}") for c in CODERS}
    res["majority"] = evaluate(mm, lambda l: f"{l}_majority")
    pd.concat(res, names=["coder", "label"]).to_csv(f"{rg.OUT_DIR}/{v}_{k}shot_evaluation.csv")

## Claude runs

In [16]:
%pip install -q anthropic pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os, getpass
os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Anthropic API key: ')

In [2]:
import run_grid_claude as rg
from run_grid_claude import *
validation = load_validation()
print(MODEL, PROMPT_HASH, len(validation), 'paragraphs')

claude-haiku-5-5 17236d3d1a9c 250 paragraphs


In [5]:
# quick check: 5 paragraphs, not saved
test = run_cell(validation.head(5), 'main', 0, save=False)
test[LABELS + ['problem']]

main, 0-shot:   0%|          | 0/5 [00:00<?, ?it/s]

,moral_rhetoric,care,fairness,loyalty,authority,sanctity,problem
0,0,0,0,0,0,0,None
1,1,0,0,0,1,0,None
2,1,1,0,1,0,0,None
3,1,1,0,1,0,0,None
4,0,0,0,0,0,0,None


In [6]:
# all 9 cells: 3 variants x 0/2/5-shot (needs EXAMPLES_FILE for 2- and 5-shot)
all_pred = run_grid(validation)
all_pred.groupby(['variant', 'shots'])['problem'].value_counts(dropna=False)


main, 0-shot:   0%|          | 0/250 [00:00<?, ?it/s]

main, 2-shot:   0%|          | 0/250 [00:00<?, ?it/s]

main, 5-shot:   0%|          | 0/250 [00:00<?, ?it/s]

attempt 1 failed: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': 'This request would exceed your rate limit of 500,000 input tokens per minute (org: dd3b2bd4-1dab-49dc-926e-93eb471af72a, model: claude-haiku-5-5). For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Reduce the prompt length or the maximum tokens requested, or try again later. View your current limits at https://console.anthropic.com/settings/limits. You may also contact sales at https://claude.com/contact-sales to discuss your options for a rate limit increase.'}, 'request_id': 'req_011ChQgaAaKkeCa8z1aYBith'}
attempt 1 failed: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': 'This request would exceed your rate limit of 500,000 input tokens per minute (org: dd3b2bd4-1dab-49dc-926e-93eb471af72a, model: claude-haiku-5-5). For details, refer to: https://docs.claude.com/en/api/rate-limi

paraphrase, 0-shot:   0%|          | 0/250 [00:00<?, ?it/s]

paraphrase, 2-shot:   0%|          | 0/250 [00:00<?, ?it/s]

... (4 outputs left out)

reverse, 2-shot:   0%|          | 0/250 [00:00<?, ?it/s]

reverse, 5-shot:   0%|          | 0/250 [00:00<?, ?it/s]

attempt 1 failed: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': 'This request would exceed your rate limit of 500,000 input tokens per minute (org: dd3b2bd4-1dab-49dc-926e-93eb471af72a, model: claude-haiku-5-5). For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Reduce the prompt length or the maximum tokens requested, or try again later. View your current limits at https://console.anthropic.com/settings/limits. You may also contact sales at https://claude.com/contact-sales to discuss your options for a rate limit increase.'}, 'request_id': 'req_011ChQh7pPmm5B8NDFPDw6wm'}
attempt 1 failed: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': 'This request would exceed your rate limit of 500,000 input tokens per minute (org: dd3b2bd4-1dab-49dc-926e-93eb471af72a, model: claude-haiku-5-5). For details, refer to: https://docs.claude.com/en/api/rate-limi

variant     shots  problem                 
main        0      NaN                         220
                   foundation_without_moral     22
                   moral_without_foundation      8
            2      NaN                         220
                   foundation_without_moral     22
                   moral_without_foundation      8
            5      NaN                         205
                   foundation_without_moral     33
                   moral_without_foundation     12
paraphrase  0      NaN                         231
                   foundation_without_moral     13
                   moral_without_foundation      6
            2      NaN                         225
                   foundation_without_moral     19
                   moral_without_foundation      6
            5      NaN                         214
                   foundation_without_moral     31
                   moral_without_foundation      5
reverse     0      NaN                

In [7]:
import re, json
bad = all_pred[(all_pred["problem"] == "parse_error") & (all_pred["variant"] == "main")]
for t in bad["raw"].head(5):
    m = re.search(r"\{.*\}", t or "", re.DOTALL)
    try:
        json.loads(m.group(0)); print("OK?")
    except Exception as e:
        print(type(e).__name__, e)
        print("len:", len(t), "| tail:", repr(t[-150:]), "\n---")

In [9]:
print(rg.PARAMS)
bad = test[test["problem"] == "parse_error"]
for _, r in bad.iterrows():
    t = r["raw"] or ""
    print("warnings:", r["warnings"], "| len:", len(t))
    print("tail:", repr(t[-200:]))
    try:
        import re, json
        json.loads(re.search(r"\{.*\}", t, re.DOTALL).group(0))
    except Exception as e:
        print("error:", type(e).__name__, e)
    print("---")

{'model': 'claude-haiku-5-5', 'max_tokens': 1024, 'thinking': {'type': 'disabled'}, 'output_config': {'effort': 'medium'}}


In [8]:
import pandas as pd
import run_grid_claude as rg
from sklearn.metrics import precision_recall_fscore_support, cohen_kappa_score

EVAL_LABELS = ["moral"] + rg.FOUNDATIONS
CODERS = ["Yunfan", "Wiktoria", "Carlos"]
LLM_COL = {"moral": "moral_rhetoric", **{f: f for f in rg.FOUNDATIONS}}

def evaluate(df, human_col_fn, llm_col_map=LLM_COL):
    rows = []
    for l in EVAL_LABELS:
        sub = df[[human_col_fn(l), llm_col_map[l]]].dropna()
        y, yhat = sub.iloc[:, 0].astype(int), sub.iloc[:, 1].astype(int)
        p, r, f, _ = precision_recall_fscore_support(y, yhat, average="binary", zero_division=0)
        rows.append({"label": l, "n": len(sub), "human_n": y.sum(), "llm_n": yhat.sum(),
                     "accuracy": (y == yhat).mean(), "precision": p, "recall": r, "f1": f,
                     "kappa": cohen_kappa_score(y, yhat)})
    return pd.DataFrame(rows).set_index("label").round(3)

In [9]:
for (v, k), g in all_pred.groupby(["variant", "shots"]):
    mm = validation.merge(g.drop(columns=["text"], errors="ignore"), on="para_id")
    mm = mm[mm["moral_rhetoric"].notna()]
    for l in EVAL_LABELS:
        mm[f"{l}_majority"] = (mm[[f"{l}_{c}" for c in CODERS]].sum(axis=1) >= 2).astype(int)
    res = {c: evaluate(mm, lambda l, c=c: f"{l}_{c}") for c in CODERS}
    res["majority"] = evaluate(mm, lambda l: f"{l}_majority")
    pd.concat(res, names=["coder", "label"]).to_csv(f"{rg.OUT_DIR}/{v}_{k}shot_evaluation.csv")